In [6]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split # <-- Nova importação adicionada

print("🚀 Iniciando o Modelo Baseline do Coordenador...")

# 1. CARREGAMENTO E MERGE
df_pessoal = pd.read_csv("base_bens.csv")
df_regional = pd.read_csv("base_regional.csv")
df_bens = pd.read_csv("base_bens.csv")
df_financeiro = pd.read_csv("base_financeiro.csv")
df_scores = pd.read_csv("base_scores.csv")
df_target = pd.read_csv("base_target.csv")

# Unindo todas as tabelas (incluindo o target para garantir que as linhas não fiquem desalinhadas)
df_completo = df_pessoal.merge(df_regional, on='SK_ID_CURR') \
                        .merge(df_bens, on='SK_ID_CURR') \
                        .merge(df_financeiro, on='SK_ID_CURR') \
                        .merge(df_scores, on='SK_ID_CURR') \
                        .merge(df_target, on='SK_ID_CURR')

# Separando X (features) e y (alvo)
y_completo = df_completo['TARGET_CREDIT_LIMIT']
X_completo = df_completo.drop(columns=['TARGET_CREDIT_LIMIT'])

# 2. PRÉ-PROCESSAMENTO BÁSICO
def limpar_dados(df):
    df_clean = df.copy()
    
    # Removendo a chave primária
    if 'SK_ID_CURR' in df_clean.columns:
        df_clean = df_clean.drop(columns=['SK_ID_CURR'])
        
    # Tratando o tempo de emprego e idade (usando if para evitar erros)
    if 'DAYS_EMPLOYED' in df_clean.columns:
        df_clean['DAYS_EMPLOYED'] = df_clean['DAYS_EMPLOYED'].replace(365243, 0)
        df_clean['ANOS_EMPREGO'] = abs(df_clean['DAYS_EMPLOYED']) / 365
        df_clean = df_clean.drop(columns=['DAYS_EMPLOYED'])
        
    if 'DAYS_BIRTH' in df_clean.columns:
        df_clean['IDADE_ANOS'] = abs(df_clean['DAYS_BIRTH']) / 365
        df_clean = df_clean.drop(columns=['DAYS_BIRTH'])
    
    # Preenchendo Nulos numéricos com a mediana
    num_cols = df_clean.select_dtypes(include=['float64', 'int64']).columns
    df_clean[num_cols] = df_clean[num_cols].fillna(df_clean[num_cols].median())
    
    # Preenchendo Nulos categóricos
    cat_cols = df_clean.select_dtypes(include=['object']).columns
    df_clean[cat_cols] = df_clean[cat_cols].fillna('Desconhecido')
    
    # Transformando texto em números (Label Encoding)
    for col in cat_cols:
        le = LabelEncoder()
        df_clean[col] = le.fit_transform(df_clean[col].astype(str))
        
    return df_clean

print("🧹 Limpando dados e corrigindo armadilhas...")
X_completo_clean = limpar_dados(X_completo)

# 3. DIVISÃO DOS DADOS PARA AVALIAÇÃO INTERNA
# Vamos usar 80% dos dados para treinar e esconder 20% para testar depois
X_treino, X_val, y_treino, y_val = train_test_split(X_completo_clean, y_completo, test_size=0.2, random_state=42)

# 4. TREINAMENTO DO MODELO
print("🧠 Treinando a Random Forest (Isso pode levar alguns segundos)...")
modelo = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
modelo.fit(X_treino, y_treino)

# 5. PREVISÃO E AVALIAÇÃO
print("🎯 Fazendo previsões...")
# Prevendo na parte de validação (dados que o modelo não viu)
previsoes_val = modelo.predict(X_val)

# Prevendo nos dados de treino (dados que o modelo já viu) - para checar Overfitting
previsoes_treino = modelo.predict(X_treino)

# Calculando as métricas
rmse_treino = np.sqrt(mean_squared_error(y_treino, previsoes_treino))
mae_treino = mean_absolute_error(y_treino, previsoes_treino)

mse_val = mean_squared_error(y_val, previsoes_val)
rmse_val = np.sqrt(mse_val)
mae_val = mean_absolute_error(y_val, previsoes_val)

print("-" * 50)
print("📊 RESULTADOS DO MODELO BASELINE:")
print(f"--- PERFORMANCE NO TREINO (80% da base) ---")
print(f"RMSE (Treino):          R$ {rmse_treino:,.2f}")
print(f"MAE  (Treino):          R$ {mae_treino:,.2f}")
print("")
print(f"--- PERFORMANCE NA VALIDAÇÃO (20% da base) ---")
print(f"Erro Quadrático (MSE):  {mse_val:,.2f}")
print(f"Raiz do Erro (RMSE):    R$ {rmse_val:,.2f}")
print(f"Erro Absoluto (MAE):    R$ {mae_val:,.2f}")
print("-" * 50)
print("💡 DICA: O valor real da qualidade do modelo é o da VALIDAÇÃO.")
print("Se o erro do Treino for *muito menor* que o erro da Validação, o modelo decorou os dados (Overfitting)!")

🚀 Iniciando o Modelo Baseline do Coordenador...
🧹 Limpando dados e corrigindo armadilhas...
🧠 Treinando a Random Forest (Isso pode levar alguns segundos)...
🎯 Fazendo previsões...
--------------------------------------------------
📊 RESULTADOS DO MODELO BASELINE:
--- PERFORMANCE NO TREINO (80% da base) ---
RMSE (Treino):          R$ 6,372.94
MAE  (Treino):          R$ 4,451.38

--- PERFORMANCE NA VALIDAÇÃO (20% da base) ---
Erro Quadrático (MSE):  130,712,165.78
Raiz do Erro (RMSE):    R$ 11,432.94
Erro Absoluto (MAE):    R$ 7,470.60
--------------------------------------------------
💡 DICA: O valor real da qualidade do modelo é o da VALIDAÇÃO.
Se o erro do Treino for *muito menor* que o erro da Validação, o modelo decorou os dados (Overfitting)!
